In [1]:
import pandas as pd
import numpy as np
import mlflow
import time
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_tracking_uri("sqlite:///../mlflow.db")  # File-based tracking is now in "maintenance mode", using SQLite instead
import pathlib
artifact_path = pathlib.Path('../mlruns').resolve().as_uri()
try:
    exp_id = mlflow.create_experiment('baseline_models', artifact_location=artifact_path)
except mlflow.exceptions.MlflowException:
    exp_id = mlflow.get_experiment_by_name('baseline_models').experiment_id
mlflow.set_experiment(experiment_id=exp_id)


c:\Users\Mustafa\s_env\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\Mustafa\s_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/07/09 12:16:05 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/09 12:16:05 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location='file:///C:/Users/Mustafa/Downloads/Staj/mlruns', creation_time=1783588567266, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783588567266, lifecycle_stage='active', name='baseline_models', tags={}, trace_location=None, workspace='default'>

In [2]:
train_df = pd.read_json("../data/processed/train.jsonl", lines=True)
val_df = pd.read_json("../data/processed/val.jsonl", lines=True)

train_df["body_clean"] = train_df["body"].fillna("")
val_df["body_clean"] = val_df["body"].fillna("")

TASK_COLS = ["type", "queue", "category", "priority"]
print("Train:", len(train_df), "| Val:", len(val_df))

Train: 22927 | Val: 4912


In [3]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(train_df["body_clean"])
X_val = vectorizer.transform(val_df["body_clean"])
print("TF-IDF sekil:", X_train.shape)

TF-IDF sekil: (22927, 20000)


In [4]:
from sklearn.preprocessing import LabelEncoder


all_results = []

for task in TASK_COLS:
    y_train = train_df[task]
    y_val = val_df[task]

    models = {
        "LogReg": LogisticRegression(max_iter=1000, random_state=42),
        "LogReg_balanced": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        "LinearSVC": LinearSVC(max_iter=2000, random_state=42),
        "LinearSVC_balanced": LinearSVC(max_iter=2000, class_weight="balanced", random_state=42),
        "MultinomialNB": MultinomialNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1)
    }

    for name, model in models.items():
        with mlflow.start_run(run_name=f"{task}_{name}"):
            if name == "XGBoost":
                le = LabelEncoder()
                y_train_encoded = le.fit_transform(y_train)
                model.fit(X_train, y_train_encoded)
                preds_encoded = model.predict(X_val)
                preds = le.inverse_transform(preds_encoded)
            else:
                model.fit(X_train, y_train)
                preds = model.predict(X_val)

                acc = accuracy_score(y_val, preds)
                f1_macro = f1_score(y_val, preds, average="macro")
                f1_weighted = f1_score(y_val, preds, average="weighted")

                mlflow.log_param("task", task)
                mlflow.log_param("model", name)
                mlflow.log_metric("accuracy", acc)
                mlflow.log_metric("f1_macro", f1_macro)
                mlflow.log_metric("f1_weighted", f1_weighted)

                labels_sorted = sorted(y_val.unique())
                cm = confusion_matrix(y_val, preds, labels=labels_sorted)
                fig, ax = plt.subplots(figsize=(6, 5))
                sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                            xticklabels=labels_sorted, yticklabels=labels_sorted, ax=ax)
                plt.title(f"{task} - {name}")
                plt.xticks(rotation=45, ha="right")
                plt.tight_layout()
                fig.savefig("cm_temp.png")
                mlflow.log_artifact("cm_temp.png")
                plt.close(fig)
                if os.path.exists('cm_temp.png'):
                    os.remove('cm_temp.png')

        all_results.append({
            "task": task, "model": name,
            "accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted
        })

results_df = pd.DataFrame(all_results)

c:\Users\Mustafa\s_env\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:22:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Mustafa\s_env\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:39:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Mustafa\s_env\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:52:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Mustafa\s_env\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:02:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, ite

In [5]:
pd.set_option("display.width", 120)
for task in TASK_COLS:
    print(f"=== {task} ===")
    print(results_df[results_df["task"] == task]
          .sort_values("f1_macro", ascending=False)
          .to_string(index=False))
    print()

=== type ===
task              model  accuracy  f1_macro  f1_weighted
type          LinearSVC  0.843037  0.844930     0.840893
type LinearSVC_balanced  0.838966  0.843490     0.838746
type    LogReg_balanced  0.822068  0.829020     0.823721
type             LogReg  0.829805  0.818158     0.822004
type       RandomForest  0.805985  0.759161     0.777443
type   GradientBoosting  0.774023  0.726063     0.746369
type            XGBoost  0.774023  0.726063     0.746369
type       DecisionTree  0.733103  0.715537     0.733319
type      MultinomialNB  0.762622  0.696433     0.715955

=== queue ===
 task              model  accuracy  f1_macro  f1_weighted
queue          LinearSVC  0.568404  0.606797     0.568722
queue LinearSVC_balanced  0.564129  0.601548     0.562643
queue       RandomForest  0.565757  0.563477     0.553267
queue    LogReg_balanced  0.489210  0.529816     0.489117
queue             LogReg  0.508143  0.516956     0.496873
queue   GradientBoosting  0.453176  0.462427     0.421